In [1]:
import os, json
import pandas as pd


In [2]:
ROOT = "/Users/ipekgezer/dataverse_files"

OUT_COUNTRY_YEAR  = "country_year_women_ratio.csv"
OUT_COUNTRY_TOTAL = "country_total_women_ratio.csv"
OUT_QUALITY       = "data_quality_report.csv"


def parse_year_from_news(news_obj): #extracting year from the 'date' field; return None if it fails
    val = news_obj.get("date")
    if not val:
        return None
    dt = pd.to_datetime(val, errors="coerce")  # invalid date strings become NaT instead of crashing
    if pd.isna(dt):
        return None
    return int(dt.year)


def to_int(x): #try to convert an id to integer
    try:
        return int(str(x))
    except:
        return None


def nearest_year_linear(target_key, key_year_pairs): #return the year of the closest key 
    best_year = None
    best_diff = None
    for k, y in key_year_pairs:
        d = abs(k - target_key)
        if best_diff is None or d < best_diff:
            best_diff = d
            best_year = y
    return best_year


rows = []
quality_rows = []

for folder in os.listdir(ROOT):
    full_path = os.path.join(ROOT, folder)
    if not os.path.isdir(full_path):
        continue

    country = folder[:3]  # country code from folder name (first 3 chars)
    news_path = os.path.join(full_path, "news.jsonl")
    img_path  = os.path.join(full_path, "images.jsonl")

    if not (os.path.exists(news_path) and os.path.getsize(news_path) > 0):
        continue
    if not (os.path.exists(img_path) and os.path.getsize(img_path) > 0):
        continue

    # Read news lines and store (id, year, line_no) for year lookup.
    news_entries = []
    with open(news_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except:
                continue

            nid = obj.get("id") 
            if nid is None:
                continue
            nid = str(nid)

            year = parse_year_from_news(obj)
            news_entries.append((nid, year, line_no))

    if not news_entries:
        continue

    # computing mode from known years in this country's news file
    known_years = [y for (_, y, _) in news_entries if y is not None] #only years
    mode_year = int(pd.Series(known_years).mode().iloc[0]) if known_years else None
    missing_year_count = sum(1 for (_, y, _) in news_entries if y is None)

    # keep only (line_no, year) pairs where year is known for nearest-line imputation
    known_line_year = [(ln, y) for (_, y, ln) in news_entries if y is not None] # years and their line num

    #building id/year map fill missing news years using nearest known line or mode.
    id_to_year = {}
    imputed_news_neighbor = 0
    for nid, y, ln in news_entries:
        if y is not None:
            id_to_year[nid] = y
        else:
            y2 = nearest_year_linear(ln, known_line_year)
            if y2 is None:
                y2 = mode_year  # if no nearby known year exists, use mode_year as fallback
            else:
                imputed_news_neighbor += 1
            if y2 is not None:
                id_to_year[nid] = int(y2)

    # prepare numeric (id_int, year) pairs for unmatched image ids 
    numeric_id_year = []
    for nid, y in id_to_year.items():
        nid_int = to_int(nid)
        if nid_int is not None:
            numeric_id_year.append((nid_int, y))

    # read images and aggregate counts 
    per_id = {}
    unmatched_images = 0
    imputed_by_numeric_nearest = 0
    imputed_by_mode_year = 0

    with open(img_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except:
                continue

            nid =  obj.get("news-id") 
            if nid is None:
                continue
            nid = str(nid)

            # male/female counts; fall back to 0 if missing or invalid.
            try:
                male = int(obj.get("male-count", obj.get("male_count", 0)) or 0)
            except:
                male = 0
            try:
                female = int(obj.get("female-count", obj.get("female_count", 0)) or 0)
            except:
                female = 0

            people = male + female
            if people <= 0:
                continue  # skip images with no detected people

            # assign a year 
            if nid in id_to_year:
                year = id_to_year[nid]
            else:
                unmatched_images += 1

                year = None
                nid_int = to_int(nid)
                if nid_int is not None:
                    year = nearest_year_linear(nid_int, numeric_id_year)
                    if year is not None:
                        imputed_by_numeric_nearest += 1

                if year is None:
                    year = mode_year
                    if year is not None:
                        imputed_by_mode_year += 1

                if year is None:
                    continue  # drop if we still cannot infer a year

            # add totals for this news_id across its people-containing images.
            if nid not in per_id:
                per_id[nid] = {"male_total": 0, "female_total": 0, "n_images_people": 0, "year": year}

            per_id[nid]["male_total"] += male
            per_id[nid]["female_total"] += female
            per_id[nid]["n_images_people"] += 1

    # one row per news_id with final totals and female_ratio.
    for nid, agg in per_id.items():
        male_total = agg["male_total"]
        female_total = agg["female_total"]
        people_total = male_total + female_total
        if people_total <= 0:
            continue

        rows.append({
            "country": country,
            "year": agg["year"],
            "news_id": nid,
            "male_total": male_total,
            "female_total": female_total,
            "people_total": people_total,
            "female_ratio": female_total / people_total,
            "n_images_people": agg["n_images_people"]
        })

    # to see the edge cases a data quality table
    quality_rows.append({
        "country": country,
        "mode_year": mode_year,
        "missing_year_count": missing_year_count,
        "imputed_news_neighbor_count": imputed_news_neighbor,
        "unmatched_images_count": unmatched_images,
        "imputed_by_numeric_nearest_count": imputed_by_numeric_nearest,
        "imputed_by_mode_year_count": imputed_by_mode_year
    })


# combine all countries into one DataFrame 
df = pd.DataFrame(rows)
if df.empty:
    raise ValueError("No usable rows parsed. Check field names and file contents.")

# year by year
country_year = (
    df.groupby(["country", "year"], as_index=False)
      .agg(
          n_news=("news_id", "nunique"),
          n_images_people=("n_images_people", "sum"),
          male_total=("male_total", "sum"),
          female_total=("female_total", "sum"),
          people_total=("people_total", "sum"),
      )
)
country_year["female_ratio"] = country_year["female_total"] / country_year["people_total"]
country_year = country_year.sort_values(["country", "year"]).reset_index(drop=True)
country_year.to_csv(OUT_COUNTRY_YEAR, index=False, encoding="utf-8")

# all covered years.
country_total = (
    country_year.groupby("country", as_index=False)
      .agg(
          year_min=("year", "min"),
          year_max=("year", "max"),
          n_news=("n_news", "sum"),
          n_images_people=("n_images_people", "sum"),
          male_total=("male_total", "sum"),
          female_total=("female_total", "sum"),
          people_total=("people_total", "sum"),
      )
)
country_total["female_ratio"] = country_total["female_total"] / country_total["people_total"]
country_total = country_total.sort_values("country").reset_index(drop=True)
country_total.to_csv(OUT_COUNTRY_TOTAL, index=False, encoding="utf-8")

quality_df = pd.DataFrame(quality_rows).sort_values("country").reset_index(drop=True)
quality_df.to_csv(OUT_QUALITY, index=False, encoding="utf-8")

print("Saved:", OUT_COUNTRY_YEAR, OUT_COUNTRY_TOTAL, OUT_QUALITY)

Saved: country_year_women_ratio.csv country_total_women_ratio.csv data_quality_report.csv


In [3]:
CY_PATH = "country_year_women_ratio.csv"
VDEM_PATH = "/Users/ipekgezer/V-Dem-CY-Full+Others-v15.csv"

OUT_PATH = "country_year_women_ratio_vdem.csv"

cy = pd.read_csv(CY_PATH)

vdem_cols = ["country_text_id", "year", "v2x_polyarchy"]  # add more later if needed
vdem = pd.read_csv(VDEM_PATH, usecols=vdem_cols, low_memory=False)

# merge on (country, year)
cy_vdem = cy.merge(vdem,left_on=["country", "year"],right_on=["country_text_id", "year"],how="left")

# delete duplicate merge key column
cy_vdem = cy_vdem.drop(columns=["country_text_id"])

# save
cy_vdem.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("Saved:", OUT_PATH)


Saved: country_year_women_ratio_vdem.csv


In [4]:
CYVDEM_PATH = "country_year_women_ratio_vdem.csv"
DQ_PATH = "data_quality_report.csv"

cyv = pd.read_csv(CYVDEM_PATH)
dq = pd.read_csv(DQ_PATH)

post = cyv[cyv["year"] >= 2000].copy()

diag = (post.groupby("country", as_index=False)
          .agg(
              cy_rows_post2000=("year", "size"),
              cy_years_post2000=("year", "nunique"),
              people_total_post2000=("people_total", "sum"),
              images_people_post2000=("n_images_people", "sum"),
              has_any_people_post2000=("people_total", lambda s: (s.fillna(0) > 0).any()),
              vdem_match_rate_post2000=("v2x_polyarchy", lambda s: s.notna().mean()),
              has_any_vdem_post2000=("v2x_polyarchy", lambda s: s.notna().any()),
          ))

dq2 = dq.merge(diag, on="country", how="left")
dq2.to_csv("data_quality_report_augmented.csv", index=False, encoding="utf-8")
print("Saved: data_quality_report_augmented.csv")


Saved: data_quality_report_augmented.csv


In [5]:
CY_PATH = "country_year_women_ratio.csv"
VDEM_PATH = "/Users/ipekgezer/V-Dem-CY-Full+Others-v15.csv"
CUTOFF_YEAR = 2000

cy = pd.read_csv(CY_PATH)
vdem = pd.read_csv(VDEM_PATH, usecols=["country_text_id", "year", "v2x_polyarchy"], low_memory=False)

merged = cy.merge(
    vdem,
    left_on=["country", "year"],
    right_on=["country_text_id", "year"],
    how="left"
)

# for the report
n_cy_all = len(cy)

cy_post = cy[cy["year"] >= CUTOFF_YEAR].copy()
n_countries_post_panel = cy_post["country"].nunique()

ymin_post = int(cy_post["year"].min())
ymax_post = int(cy_post["year"].max())

share_people_gt0 = (cy["people_total"] > 0).mean()

vdem_match_rate = 1.0 - merged["v2x_polyarchy"].isna().mean()


# Countries used in hypothesis test 
post = merged[merged["year"] >= CUTOFF_YEAR].copy()

countries_panel = set(post["country"].dropna().unique())

eligible = post.dropna(subset=["female_ratio", "v2x_polyarchy", "people_total", "female_total"]).copy()
eligible = eligible[eligible["people_total"] > 0].copy()

countries_test = set(eligible["country"].unique())

dropped = sorted(list(countries_panel - countries_test))

# drop reasons per dropped country
reason_rows = []
for c in dropped:
    g = post[post["country"] == c].copy()
    has_people = (g["people_total"].fillna(0) > 0).any()
    has_vdem = g["v2x_polyarchy"].notna().any()

    if not has_people:
        reason = "No people_total>0 in post-2000"
    elif not has_vdem:
        reason = "No V-Dem match in post-2000"
    else:
        reason = "Dropped by missingness (female_ratio/female_total/etc.)"

    reason_rows.append({"country": c, "drop_reason_post2000": reason})

drop_df = pd.DataFrame(reason_rows)


# summary table 
print("Country-year observations (all years):", n_cy_all)
print("Countries observed (post-2000 panel):", n_countries_post_panel)
print("Years covered (post-2000):", f"{ymin_post}-{ymax_post}")
print("Share with people_total>0:", round(share_people_gt0, 4))
print("V-Dem match rate:", round(vdem_match_rate, 4))

print("\nCountries used in hypothesis test:", len(countries_test))
print("Countries dropped:", len(dropped))

if len(drop_df) > 0:
    print("\nDrop reasons:")
    print(drop_df["drop_reason_post2000"].value_counts().to_string())

    print("\nDropped country codes:")
    print(", ".join(dropped))


Country-year observations (all years): 1359
Countries observed (post-2000 panel): 143
Years covered (post-2000): 2000-2024
Share with people_total>0: 1.0
V-Dem match rate: 0.9536

Countries used in hypothesis test: 137
Countries dropped: 6

Drop reasons:
drop_reason_post2000
No V-Dem match in post-2000    6

Dropped country codes:
AND, BHS, BLZ, BRN, VCT, WSM
